In [1]:
import subprocess
import uproot
import os
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import matplotlib.pylab as plt
import awkward as ak
import vector
import json
import boost_histogram as bh

vector.register_awkward()

In [2]:
SAMPLES = {
    "DY_inclusive_NLO": {"recid": 35669, "is_data": False, "is_dy": True, "group": "DY"},
    # "DY_inclusive_LO":  {"recid": 35671, "is_data": False, "is_dy": True, "group": "DY"},

    "WW_2L2Nu":     {"recid": 72676, "is_data": False, "is_dy": False, "group": "diboson"},
    "WZ_3LNu":      {"recid": 72752, "is_data": False, "is_dy": False, "group": "diboson"},
    "WZ_2Q2L":      {"recid": 72742, "is_data": False, "is_dy": False, "group": "diboson"},
    "ZZ_2L2Nu":     {"recid": 75567, "is_data": False, "is_dy": False, "group": "diboson"},
    "ZZ_2Q2L":      {"recid": 75573, "is_data": False, "is_dy": False, "group": "diboson"},
    "ZZ_4L":        {"recid": 75589, "is_data": False, "is_dy": False, "group": "diboson"},

    # ttbar -- powheg, split by decay channel (mutually exclusive, don't add TTJets amcatnlo too)
    "TTTo2L2Nu":       {"recid": 67801, "is_data": False, "is_dy": False, "group": "ttbar"},
    "TTToSemiLeptonic": {"recid": 67993, "is_data": False, "is_dy": False, "group": "ttbar"},
    "TTToHadronic":    {"recid": 67841, "is_data": False, "is_dy": False, "group": "ttbar"},

    # Wjets -- amcatnloFXFX inclusive (don't also add the madgraphMLM version)
    "WJetsToLNu": {"recid": 69745, "is_data": False, "is_dy": False, "group": "wjets"},

    "Run2016G": {"recid": 30529, "is_data": True},
    "Run2016H": {"recid": 30562, "is_data": True},
}

CACHE_DIR = "/data/atlas/users/lvdurenw/nano_cache"
branches = [
    "run", "luminosityBlock", "event",
    "Electron_pt", "Electron_eta", "Electron_phi", "Electron_mass",
    "Electron_charge", "Electron_cutBased",
]
TRIGGER = "HLT_Ele27_WPTight_Gsf"
branches_mc_extra = [
    "genWeight", "L1PreFiringWeight_Nom", "L1PreFiringWeight_Up", "L1PreFiringWeight_Dn",
    "Pileup_nTrueInt",
    "LHEPart_pdgId", "LHEPart_status",
    "Electron_deltaEtaSC",
    "LHEPdfWeight", "LHEScaleWeight",
    "Electron_dEscaleUp", "Electron_dEscaleDown",
    "Electron_dEsigmaUp", "Electron_dEsigmaDown",
    "PSWeight",
]

SYST_NAMES = [
    "PUUp", "PUDown",
    "L1PrefireUp", "L1PrefireDown",
    "EleRecoUp", "EleRecoDown",
    "EleIDUp", "EleIDDown",
    "FSRUp", "FSRDown",
]

# NEW: these vary the electron 4-vector (hence the mass), not just the event weight,
# so they're tracked separately from SYST_NAMES.
SHAPE_SYST_NAMES = ["EESUp", "EESDown", "EERUp", "EERDown"]

def human_size(n_bytes):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n_bytes < 1024:
            return f"{n_bytes:.1f} {unit}"
        n_bytes /= 1024
    return f"{n_bytes:.1f} PB"

In [3]:
XSEC_PB = {
    "DY_inclusive_NLO": 6077.22,
    # "DY_inclusive_LO":  6077.22,

    "WW_2L2Nu": 12.178,
    "WZ_3LNu":  4.42965,
    "WZ_2Q2L":  5.595,
    "ZZ_2L2Nu": 0.564,
    "ZZ_2Q2L":  3.688,
    "ZZ_4L":    1.256,

    # ttbar (831.76 pb inclusive, split by BR: dilep 10.6%, semilep 43.9%, had 45.4%)
    "TTTo2L2Nu":        88.29,
    "TTToSemiLeptonic": 365.35,
    "TTToHadronic":     377.96,

    # Wjets (NNLO inclusive)
    "WJetsToLNu": 61526.7,
}

# Get this from GRL using brilcalc lumi -c web --begin 278820 --end 284044 -i datasets/GRL/GRL.txt -u /fb > log.txt
LUMI_PB = 16393.381    # normtag_PHYSICS value for these certified lumisections (CMS Open Data record 1059);
                       # the earlier 16290.713420 was brilcalc without --normtag (0.63% low)


In [4]:
import subprocess, tarfile, glob

CORR_TAR_URL = "https://opendata.cern.ch/eos/opendata/cms/corrections/jsonpog-integration-2016post.tar"
CORR_DIR = "../datasets/corrections"
os.makedirs(CORR_DIR, exist_ok=True)
tar_path = os.path.join(CORR_DIR, "jsonpog-integration-2016post.tar")

if not glob.glob(f"{CORR_DIR}/**/electron.json.gz", recursive=True):
    if not os.path.exists(tar_path):
        subprocess.run(["wget", "-q", CORR_TAR_URL, "-O", tar_path], check=True)
    with tarfile.open(tar_path) as tf:
        tf.extractall(CORR_DIR)

pu_json_path  = glob.glob(f"{CORR_DIR}/**/puWeights.json.gz", recursive=True)[0]
ele_json_path = glob.glob(f"{CORR_DIR}/**/electron.json.gz", recursive=True)[0]
print("PU json:", pu_json_path)
print("Electron json:", ele_json_path)

PU json: ../datasets/corrections/POG/LUM/2016postVFP_UL/puWeights.json.gz
Electron json: ../datasets/corrections/POG/EGM/2016postVFP_UL/electron.json.gz


In [5]:
import gzip
import correctionlib

def load_correction_set(path):
    with gzip.open(path, "rt") as f:
        data = f.read().strip()
    return correctionlib._core.CorrectionSet.from_string(data)

pu_set  = load_correction_set(pu_json_path)
pu_corr = pu_set["Collisions16_UltraLegacy_goldenJSON"]

ele_set  = load_correction_set(ele_json_path)
ele_corr = ele_set["UL-Electron-ID-SF"]

YEAR_TAG = "2016postVFP"   # matches Run2016G+H
ELE_ID_WP = "Medium"       # matches Electron_cutBased >= 3
ELE_RECO_WP = "RecoAbove20"  # matches your pt > 20 selection

In [6]:
json_path = "../datasets/GRL/GRL.txt"
json_url = "https://opendata.cern.ch/record/14220/files/Cert_271036-284044_13TeV_Legacy2016_Collisions16_JSON.txt"

if not os.path.exists(json_path):
    resp = requests.get(json_url)
    resp.raise_for_status()
    with open(json_path, "wb") as f:
        f.write(resp.content)

with open(json_path) as f:
    golden_json_raw = json.load(f)

golden_json = {int(run): ranges for run, ranges in golden_json_raw.items()}
print(f"Loaded {len(golden_json)} certified runs")

Loaded 393 certified runs


In [7]:
def lumi_mask(runs, lumis, json_dict):
    """Boolean mask: True where (run, luminosityBlock) falls in a certified range."""
    runs_np = ak.to_numpy(runs)
    lumis_np = ak.to_numpy(lumis)
    mask = np.zeros(len(runs_np), dtype=bool)

    for run in np.unique(runs_np):
        run = int(run)
        if run not in json_dict:
            continue
        run_sel = runs_np == run
        lumi_vals = lumis_np[run_sel]
        run_mask = np.zeros(len(lumi_vals), dtype=bool)
        for lo, hi in json_dict[run]:
            run_mask |= (lumi_vals >= lo) & (lumi_vals <= hi)
        mask[run_sel] = run_mask

    return mask

In [8]:
for name, info in SAMPLES.items():
    result = subprocess.run(
        ["cernopendata-client", "get-file-locations", "--recid", str(info["recid"]), "--protocol", "http"],
        capture_output=True, text=True, check=True,
    )
    info["file_list"] = [f for f in result.stdout.strip().splitlines() if f]
    print(f"{name}: {len(info['file_list'])} files (recid {info['recid']})")

DY_inclusive_NLO: 41 files (recid 35669)
WW_2L2Nu: 7 files (recid 72676)
WZ_3LNu: 31 files (recid 72752)
WZ_2Q2L: 39 files (recid 72742)
ZZ_2L2Nu: 15 files (recid 75567)
ZZ_2Q2L: 14 files (recid 75573)
ZZ_4L: 99 files (recid 75589)
TTTo2L2Nu: 49 files (recid 67801)
TTToSemiLeptonic: 138 files (recid 67993)
TTToHadronic: 146 files (recid 67841)
WJetsToLNu: 28 files (recid 69745)
Run2016G: 71 files (recid 30529)
Run2016H: 80 files (recid 30562)


In [9]:
for name, info in SAMPLES.items():
    total_bytes = 0
    sized = 0
    for url in info["file_list"]:
        resp = requests.head(url, allow_redirects=True)
        if resp.status_code == 200 and "Content-Length" in resp.headers:
            total_bytes += int(resp.headers["Content-Length"])
            sized += 1
    print(f"{name}: {sized}/{len(info['file_list'])} files sized, total = {human_size(total_bytes)}")

DY_inclusive_NLO: 41/41 files sized, total = 84.9 GB
WW_2L2Nu: 7/7 files sized, total = 4.0 GB
WZ_3LNu: 31/31 files sized, total = 14.4 GB
WZ_2Q2L: 39/39 files sized, total = 20.6 GB
ZZ_2L2Nu: 15/15 files sized, total = 20.1 GB
ZZ_2Q2L: 14/14 files sized, total = 21.0 GB
ZZ_4L: 99/99 files sized, total = 66.6 GB
TTTo2L2Nu: 49/49 files sized, total = 84.9 GB
TTToSemiLeptonic: 64/138 files sized, total = 139.3 GB
TTToHadronic: 20/146 files sized, total = 24.9 GB
WJetsToLNu: 3/28 files sized, total = 2.3 GB
Run2016G: 9/71 files sized, total = 16.5 GB
Run2016H: 11/80 files sized, total = 13.4 GB


In [10]:
def download_one(url, sample_dir, retries=5):
    fname = os.path.join(sample_dir, os.path.basename(url))
    if os.path.exists(fname):
        return fname
    for attempt in range(retries):
        resp = requests.get(url, stream=True)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 10))
            time.sleep(wait)
            continue
        resp.raise_for_status()
        with open(fname, "wb") as f:
            for block in resp.iter_content(chunk_size=1 << 20):
                f.write(block)
        return fname
    raise RuntimeError(f"Failed to download {url} after {retries} attempts")

def download_sample(name, file_list, max_workers=8):
    sample_dir = os.path.join(CACHE_DIR, name)
    os.makedirs(sample_dir, exist_ok=True)

    local_paths = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(download_one, url, sample_dir): url for url in file_list}
        for i, fut in enumerate(as_completed(futures), 1):
            try:
                local_paths.append(fut.result())
                if i % 10 == 0 or i == len(file_list):
                    print(f"  {name}: [{i}/{len(file_list)}] done")
            except Exception as e:
                print(f"  {name}: FAILED {futures[fut]} — {e}")

    # preserve original file order (as_completed returns them out of order)
    ordered = [os.path.join(sample_dir, os.path.basename(u)) for u in file_list]
    return ordered

for name, info in SAMPLES.items():
    print(f"--- downloading {name} ---")
    info["local_paths"] = download_sample(name, info["file_list"])
    print(f"{name}: {len(info['local_paths'])} files cached locally")

--- downloading DY_inclusive_NLO ---
  DY_inclusive_NLO: [10/41] done
  DY_inclusive_NLO: [20/41] done
  DY_inclusive_NLO: [30/41] done
  DY_inclusive_NLO: [40/41] done
  DY_inclusive_NLO: [41/41] done
DY_inclusive_NLO: 41 files cached locally
--- downloading WW_2L2Nu ---
  WW_2L2Nu: [7/7] done
WW_2L2Nu: 7 files cached locally
--- downloading WZ_3LNu ---
  WZ_3LNu: [10/31] done
  WZ_3LNu: [20/31] done
  WZ_3LNu: [30/31] done
  WZ_3LNu: [31/31] done
WZ_3LNu: 31 files cached locally
--- downloading WZ_2Q2L ---
  WZ_2Q2L: [10/39] done
  WZ_2Q2L: [20/39] done
  WZ_2Q2L: [30/39] done
  WZ_2Q2L: [39/39] done
WZ_2Q2L: 39 files cached locally
--- downloading ZZ_2L2Nu ---
  ZZ_2L2Nu: [10/15] done
  ZZ_2L2Nu: [15/15] done
ZZ_2L2Nu: 15 files cached locally
--- downloading ZZ_2Q2L ---
  ZZ_2Q2L: [10/14] done
  ZZ_2Q2L: [14/14] done
ZZ_2Q2L: 14 files cached locally
--- downloading ZZ_4L ---
  ZZ_4L: [10/99] done
  ZZ_4L: [20/99] done
  ZZ_4L: [30/99] done
  ZZ_4L: [40/99] done
  ZZ_4L: [50/99] done

In [11]:
def get_sum_genweight(local_paths, treename="Runs", branch="genEventSumw"):
    """Sum genEventSumw across all files -- this is the FULL sample sum,
    not restricted to your selected events, and is required for correct
    MC normalization."""
    total = 0.0
    for p in local_paths:
        with uproot.open(p) as f:
            if treename not in f:
                raise KeyError(f"{treename} tree not found in {p} -- check NanoAOD version")
            total += f[treename][branch].array(library="np").sum()
    return total

for name, info in SAMPLES.items():
    if not info["is_data"]:
        info["sum_genweight"] = get_sum_genweight(info["local_paths"])
        print(f"{name}: sum_genweight = {info['sum_genweight']:.4e}")

DY_inclusive_NLO: sum_genweight = 1.2209e+12
WW_2L2Nu: sum_genweight = 3.2147e+07
WZ_3LNu: sum_genweight = 8.8368e+07
WZ_2Q2L: sum_genweight = 1.2976e+08
ZZ_2L2Nu: sum_genweight = 1.5510e+07
ZZ_2Q2L: sum_genweight = 7.5775e+07
ZZ_4L: sum_genweight = 6.9075e+07
TTTo2L2Nu: sum_genweight = 3.1401e+09
TTToSemiLeptonic: sum_genweight = 4.3548e+10
TTToHadronic: sum_genweight = 3.3609e+10
WJetsToLNu: sum_genweight = 4.8730e+12


In [12]:
branches_mc_extra += ["LHEPart_pt", "LHEPart_eta", "LHEPart_phi", "LHEPart_mass"]

def truth_dilepton_mass_and_channel(chunk):
    pdgid = chunk["LHEPart_pdgId"]
    status = chunk["LHEPart_status"]
    lep_mask = (status == 1) & ((abs(pdgid) == 11) | (abs(pdgid) == 13) | (abs(pdgid) == 15))
    two_lep = ak.sum(lep_mask, axis=1) == 2

    lhe = ak.zip({
        "pt": chunk["LHEPart_pt"][lep_mask], "eta": chunk["LHEPart_eta"][lep_mask],
        "phi": chunk["LHEPart_phi"][lep_mask], "mass": chunk["LHEPart_mass"][lep_mask],
        "pdgId": pdgid[lep_mask],
    }, with_name="Momentum4D")[two_lep]

    m_truth = ak.to_numpy((lhe[:, 0] + lhe[:, 1]).mass)
    flav = ak.to_numpy(abs(lhe[:, 0].pdgId))
    channel = np.select([flav == 11, flav == 13, flav == 15], ["ee", "mumu", "tautau"], default="other")
    return m_truth, channel, ak.to_numpy(two_lep)

def truth_window_sumgenweight(local_paths, mass_lo=60., mass_hi=120., chunk_size="200 MB"):
    sum_w = {"ee": 0.0, "mumu": 0.0, "tautau": 0.0}
    for chunk in uproot.iterate(
        [f"{p}:Events" for p in local_paths],
        filter_name=["genWeight", "LHEPart_pt", "LHEPart_eta", "LHEPart_phi",
                     "LHEPart_mass", "LHEPart_pdgId", "LHEPart_status"],
        library="ak", step_size=chunk_size,
    ):
        genw = ak.to_numpy(chunk["genWeight"])
        m_truth, channel, has_pair = truth_dilepton_mass_and_channel(chunk)
        genw_paired = genw[has_pair]
        in_window = (m_truth >= mass_lo) & (m_truth < mass_hi)
        for ch in sum_w:
            sum_w[ch] += genw_paired[(channel == ch) & in_window].sum()
    return sum_w

sum_w = truth_window_sumgenweight(SAMPLES["DY_inclusive_NLO"]["local_paths"])
xsec_incl = XSEC_PB["DY_inclusive_NLO"]
denom = SAMPLES["DY_inclusive_NLO"]["sum_genweight"]  # reuse the Runs-tree total you already computed

reference_xsec_ee = xsec_incl * sum_w["ee"] / denom
print(reference_xsec_ee)

1954.1032472811223


In [ ]:
bins = np.linspace(60, 120, 31)   # 30 bins, 2 GeV wide

In [ ]:
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

def get_total_entries(local_paths, treename="Events"):
    total = 0
    for p in local_paths:
        with uproot.open(p) as f:
            total += f[treename].num_entries
    return total

def classify_dy_channel(lhe_pdgid, lhe_status):
    outgoing = lhe_status == 1
    lep_mask = outgoing & (
        (abs(lhe_pdgid) == 11) | (abs(lhe_pdgid) == 13) | (abs(lhe_pdgid) == 15)
    )
    first_pdgid = ak.to_numpy(ak.firsts(lhe_pdgid[lep_mask]))
    channel = np.select(
        [np.abs(first_pdgid) == 11, np.abs(first_pdgid) == 13, np.abs(first_pdgid) == 15],
        ["ee", "mumu", "tautau"],
        default="other",
    )
    return channel


def pair_electron_sf(eta_sc_2d, pt_2d, valtype, wp):
    """eta_sc_2d, pt_2d: numpy arrays of shape (Nevt, 2). Returns per-event product of the
    per-electron SF (leading * subleading)."""
    nev = eta_sc_2d.shape[0]
    flat_sf = ele_corr.evalv(YEAR_TAG, valtype, wp, eta_sc_2d.reshape(-1), pt_2d.reshape(-1))
    flat_sf = np.asarray(flat_sf)
    return flat_sf.reshape(nev, 2).prod(axis=1)


def _accumulate_envelope(hist_sum, nominal_sum, mass_np, weight_np, lhe_matrix, bins):
    """Fold one chunk's (mass, nominal weight, LHE weight matrix) into running per-replica
    histogram sums, in place. hist_sum: (n_replica, n_bins) or None (allocated on first call).

    Vectorized over replicas: fold the replica index into the bin index and do a single
    bincount over Nvalid * nrep entries instead of nrep separate calls.
    """
    nrep = lhe_matrix.shape[1]
    nbins = len(bins) - 1
    if hist_sum is None:
        hist_sum = np.zeros((nrep, nbins))

    nom_counts, _ = np.histogram(mass_np, bins=bins, weights=weight_np)
    nominal_sum += nom_counts

    bin_idx = np.digitize(mass_np, bins) - 1
    valid = (bin_idx >= 0) & (bin_idx < nbins)
    bin_idx_v = bin_idx[valid].astype(np.int64)
    w_v = weight_np[valid]
    lhe_v = lhe_matrix[valid]                       # (Nvalid, nrep)

    vals = (w_v[:, None] * lhe_v).ravel()
    combined_idx = (bin_idx_v[:, None] * nrep + np.arange(nrep)).ravel()

    flat_sum = np.bincount(combined_idx, weights=vals, minlength=nbins * nrep)
    hist_sum += flat_sum.reshape(nbins, nrep).T
    return hist_sum


def process_sample(name, info, branches, bins, chunk_size="200 MB", position=0):
    """Returns nominal mass/weight/channel arrays, a dict of one-at-a-time systematic weight
    arrays (same length/order as the nominal arrays), and small PDF/scale envelope histograms
    (fixed-size arrays over `bins`, not per-event matrices).

    `position` pins this sample's tqdm bar to its own row so concurrent bars don't clobber
    each other in the notebook output.
    """
    is_data = info["is_data"]
    these_branches = branches + [TRIGGER] + ([] if is_data else branches_mc_extra)

    total_events = get_total_entries(info["local_paths"])

    masses, weights, channels = [], [], []
    syst_weights = {k: [] for k in SYST_NAMES}
    mass_syst = {k: [] for k in SHAPE_SYST_NAMES}
    pdf_hist_sum, pdf_nom_sum = None, np.zeros(len(bins) - 1)
    scale_hist_sum, scale_nom_sum = None, np.zeros(len(bins) - 1)

    chunk_iter = uproot.iterate(
        [f"{p}:Events" for p in info["local_paths"]],
        filter_name=these_branches,
        library="ak",
        step_size=chunk_size,
    )

    pbar = tqdm(total=total_events, desc=name, unit="evt", unit_scale=True,
                leave=True, position=position)
    for chunk in chunk_iter:
        n_raw = len(chunk)

        trig_mask = chunk[TRIGGER]
        if is_data:
            good_lumi = lumi_mask(chunk["run"], chunk["luminosityBlock"], golden_json)
            event_mask = trig_mask & good_lumi
        else:
            event_mask = trig_mask

        chunk = chunk[event_mask]

        good_e = (chunk["Electron_pt"] > 20) & (abs(chunk["Electron_eta"]) < 2.5) & (chunk["Electron_cutBased"] >= 3)
        n_good = ak.sum(good_e, axis=1)
        two_e_mask = n_good == 2

        sel = chunk[two_e_mask]
        good_mask_sel = good_e[two_e_mask]

        ele = ak.zip({
            "pt": sel["Electron_pt"][good_mask_sel],
            "eta": sel["Electron_eta"][good_mask_sel],
            "phi": sel["Electron_phi"][good_mask_sel],
            "mass": sel["Electron_mass"][good_mask_sel],
            "charge": sel["Electron_charge"][good_mask_sel],
        }, with_name="Momentum4D")

        if not is_data:
            eta_sc_all = sel["Electron_eta"][good_mask_sel] + sel["Electron_deltaEtaSC"][good_mask_sel]

        oppsign = ele[:, 0].charge * ele[:, 1].charge == -1
        ele = ele[oppsign]
        if not is_data:
            eta_sc = eta_sc_all[oppsign]

            # NEW: EES/ER -- reuses the already-selected pair, only pt is shifted (doesn't
            # redo the pt>20 selection with shifted pt; fine for small shifts, but redo the
            # selection too if EES/ER turn out to be large enough to move electrons across
            # the pt threshold).
            for shift_key, field in [("EESUp", "Electron_dEscaleUp"), ("EESDown", "Electron_dEscaleDown"),
                                      ("EERUp", "Electron_dEsigmaUp"), ("EERDown", "Electron_dEsigmaDown")]:
                pt_shift = sel[field][good_mask_sel][oppsign]
                ele_shift = ak.zip({
                    "pt": ele.pt + pt_shift, "eta": ele.eta, "phi": ele.phi, "mass": ele.mass,
                }, with_name="Momentum4D")
                mass_syst[shift_key].append(ak.to_numpy((ele_shift[:, 0] + ele_shift[:, 1]).mass))

        m = (ele[:, 0] + ele[:, 1]).mass
        m_np = ak.to_numpy(m)
        masses.append(m_np)

        if is_data:
            weights.append(np.ones(len(m)))
            channels.append(np.full(len(m), "data"))
            pbar.update(n_raw)
            continue

        if len(m_np) == 0:
            channels.append(np.array([], dtype=object))
            for k in SYST_NAMES:
                syst_weights[k].append(np.array([]))
            for k in SHAPE_SYST_NAMES:
                mass_syst[k].append(np.array([]))
            pbar.update(n_raw)
            continue

        genw_np     = ak.to_numpy(sel["genWeight"][oppsign])
        prefire_nom = ak.to_numpy(sel["L1PreFiringWeight_Nom"][oppsign])
        prefire_up  = ak.to_numpy(sel["L1PreFiringWeight_Up"][oppsign])
        prefire_dn  = ak.to_numpy(sel["L1PreFiringWeight_Dn"][oppsign])
        nTrueInt_np = ak.to_numpy(sel["Pileup_nTrueInt"][oppsign])

        pu_nom = np.asarray(pu_corr.evalv(nTrueInt_np, "nominal"))
        pu_up  = np.asarray(pu_corr.evalv(nTrueInt_np, "up"))
        pu_dn  = np.asarray(pu_corr.evalv(nTrueInt_np, "down"))

        eta_np = ak.to_numpy(eta_sc)   # (N, 2)
        pt_np  = ak.to_numpy(ele.pt)   # (N, 2)

        reco_sf_nom = pair_electron_sf(eta_np, pt_np, "sf",     ELE_RECO_WP)
        reco_sf_up  = pair_electron_sf(eta_np, pt_np, "sfup",   ELE_RECO_WP)
        reco_sf_dn  = pair_electron_sf(eta_np, pt_np, "sfdown", ELE_RECO_WP)

        id_sf_nom = pair_electron_sf(eta_np, pt_np, "sf",     ELE_ID_WP)
        id_sf_up  = pair_electron_sf(eta_np, pt_np, "sfup",   ELE_ID_WP)
        id_sf_dn  = pair_electron_sf(eta_np, pt_np, "sfdown", ELE_ID_WP)

        xsec = XSEC_PB[name]
        if xsec is None or LUMI_PB is None:
            raise ValueError(f"Set XSEC_PB['{name}'] and LUMI_PB before running")
        norm = xsec * LUMI_PB / info["sum_genweight"]
        base = genw_np * norm

        w_nom = base * prefire_nom * pu_nom * reco_sf_nom * id_sf_nom
        weights.append(w_nom)

        syst_weights["PUUp"].append(base * prefire_nom * pu_up * reco_sf_nom * id_sf_nom)
        syst_weights["PUDown"].append(base * prefire_nom * pu_dn * reco_sf_nom * id_sf_nom)
        syst_weights["L1PrefireUp"].append(base * prefire_up * pu_nom * reco_sf_nom * id_sf_nom)
        syst_weights["L1PrefireDown"].append(base * prefire_dn * pu_nom * reco_sf_nom * id_sf_nom)
        syst_weights["EleRecoUp"].append(base * prefire_nom * pu_nom * reco_sf_up * id_sf_nom)
        syst_weights["EleRecoDown"].append(base * prefire_nom * pu_nom * reco_sf_dn * id_sf_nom)
        syst_weights["EleIDUp"].append(base * prefire_nom * pu_nom * reco_sf_nom * id_sf_up)
        syst_weights["EleIDDown"].append(base * prefire_nom * pu_nom * reco_sf_nom * id_sf_dn)
        psweight_np = ak.to_numpy(sel["PSWeight"][oppsign])  # (N, 4): [ISRUp, FSRUp, ISRDown, FSRDown]
        syst_weights["FSRUp"].append(base * prefire_nom * pu_nom * reco_sf_nom * id_sf_nom * psweight_np[:, 1])
        syst_weights["FSRDown"].append(base * prefire_nom * pu_nom * reco_sf_nom * id_sf_nom * psweight_np[:, 3])

        if info.get("is_dy", False):
            ch = classify_dy_channel(sel["LHEPart_pdgId"][oppsign], sel["LHEPart_status"][oppsign])
        else:
            ch = np.full(len(m), info.get("group", name))
        channels.append(ch)

        lhepdf_np = ak.to_numpy(sel["LHEPdfWeight"][oppsign])
        lhescale_np = ak.to_numpy(sel["LHEScaleWeight"][oppsign])
        pdf_hist_sum = _accumulate_envelope(pdf_hist_sum, pdf_nom_sum, m_np, w_nom, lhepdf_np, bins)
        scale_hist_sum = _accumulate_envelope(scale_hist_sum, scale_nom_sum, m_np, w_nom, lhescale_np, bins)
        del lhepdf_np, lhescale_np

        pbar.update(n_raw)
        pbar.set_postfix(selected=sum(len(x) for x in masses))

    pbar.close()

    if not is_data and pdf_hist_sum is not None:
        # --- rescale each replica to the nominal integral so only shape varies ---
        pdf_hist_sum *= pdf_nom_sum.sum() / pdf_hist_sum.sum(axis=1, keepdims=True)
        scale_hist_sum *= scale_nom_sum.sum() / scale_hist_sum.sum(axis=1, keepdims=True)

        delta_pdf = np.sqrt(np.mean((pdf_hist_sum - pdf_nom_sum) ** 2, axis=0))
        pdf_up, pdf_down = pdf_nom_sum + delta_pdf, np.clip(pdf_nom_sum - delta_pdf, 0, None)

        delta_scale = np.maximum(scale_hist_sum.max(axis=0) - scale_nom_sum,
                                  scale_nom_sum - scale_hist_sum.min(axis=0))
        scale_up, scale_down = scale_nom_sum + delta_scale, np.clip(scale_nom_sum - delta_scale, 0, None)
    else:
        pdf_up = pdf_down = scale_up = scale_down = None

    return {
        "mass": np.concatenate(masses) if masses else np.array([]),
        "weight": np.concatenate(weights) if weights else np.array([]),
        "channel": np.concatenate(channels) if channels else np.array([]),
        "weight_syst": {k: (np.concatenate(v) if v else np.array([])) for k, v in syst_weights.items()},
        "pdf_up": pdf_up, "pdf_down": pdf_down,
        "scale_up": scale_up, "scale_down": scale_down,
        "mass_syst": {k: (np.concatenate(v) if v else np.array([])) for k, v in mass_syst.items()},
        "pdf_up": pdf_up, "pdf_down": pdf_down, "scale_up": scale_up, "scale_down": scale_down,
    }


# --- run all samples concurrently ---
# max_workers: cap this below len(SAMPLES) if you're I/O- or memory-constrained (each in-flight
# sample holds one "200 MB"-ish chunk of arrays at a time). Start conservative and raise it if
# CPU/disk have headroom -- with 14 samples and heavy correctionlib/numpy work per chunk, 4-6
# workers is a reasonable starting point rather than firing all 14 at once.
MAX_WORKERS = min(6, len(SAMPLES))

results = {}
errors = {}
print_lock = threading.Lock()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {
        ex.submit(process_sample, name, info, branches, bins, position=i): name
        for i, (name, info) in enumerate(SAMPLES.items())
    }
    for fut in as_completed(futures):
        name = futures[fut]
        try:
            results[name] = fut.result()
            with print_lock:
                tqdm.write(f"  {name}: {len(results[name]['mass'])} selected events")
        except Exception as e:
            errors[name] = e
            with print_lock:
                tqdm.write(f"  {name}: FAILED — {e}")

if errors:
    print(f"\n{len(errors)} sample(s) failed: {list(errors.keys())}")

WW_2L2Nu:   0%|          | 0.00/2.90M [00:00<?, ?evt/s]

ZZ_2Q2L:   0%|          | 0.00/13.7M [00:00<?, ?evt/s]

ZZ_2L2Nu:   0%|          | 0.00/15.9M [00:00<?, ?evt/s]

WZ_3LNu:   0%|          | 0.00/10.4M [00:00<?, ?evt/s]

WZ_2Q2L:   0%|          | 0.00/13.5M [00:00<?, ?evt/s]

DY_inclusive_NLO:   0%|          | 0.00/71.8M [00:00<?, ?evt/s]

In [ ]:
bins = np.linspace(60, 120, 31)   # 30 bins, 2 GeV wide
bin_centers = 0.5 * (bins[1:] + bins[:-1])
bin_widths = np.diff(bins)

fig, (ax, ax_ratio) = plt.subplots(
    2, 1, figsize=(10, 8), sharex=True,
    gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05},
)

dy = results["DY_inclusive_NLO"]

mc_hists = []
mc_weights_list = []
mc_names = []

# ttbar -- combined into one stack entry
ttbar_names = [n for n, info in SAMPLES.items() if info.get("group") == "ttbar"]
if ttbar_names:
    ttbar_mass = np.concatenate([results[n]["mass"] for n in ttbar_names])
    ttbar_weight = np.concatenate([results[n]["weight"] for n in ttbar_names])
    mc_hists.append(ttbar_mass)
    mc_weights_list.append(ttbar_weight)
    mc_names.append("ttbar")

# Wjets
wjets_names = [n for n, info in SAMPLES.items() if info.get("group") == "wjets"]
if wjets_names:
    wjets_mass = np.concatenate([results[n]["mass"] for n in wjets_names])
    wjets_weight = np.concatenate([results[n]["weight"] for n in wjets_names])
    mc_hists.append(wjets_mass)
    mc_weights_list.append(wjets_weight)
    mc_names.append("W+jets")

# Diboson -- combined into one stack entry
diboson_names = [n for n, info in SAMPLES.items() if info.get("group") == "diboson"]
if diboson_names:
    diboson_mass = np.concatenate([results[n]["mass"] for n in diboson_names])
    diboson_weight = np.concatenate([results[n]["weight"] for n in diboson_names])
    mc_hists.append(diboson_mass)
    mc_weights_list.append(diboson_weight)
    mc_names.append("Diboson")

# DY split by true decay channel, tautau first so ee ends up on top of the stack
for ch, label in [("tautau", "DY→ττ"), ("mumu", "DY→μμ"), ("ee", "DY→ee")]:
    mask = dy["channel"] == ch
    if mask.sum() == 0:
        continue
    mc_hists.append(dy["mass"][mask])
    mc_weights_list.append(dy["weight"][mask])
    mc_names.append(label)

if mc_hists:
    ax.hist(
        mc_hists,
        bins=bins,
        weights=mc_weights_list,
        stacked=True,
        label=mc_names,
        alpha=0.7,
    )

# --- rest of the cell (MC uncertainty band, data, ratio panel) is unchanged ---
all_mc_mass = np.concatenate(mc_hists) if mc_hists else np.array([])
all_mc_weight = np.concatenate(mc_weights_list) if mc_weights_list else np.array([])

mc_counts, _ = np.histogram(all_mc_mass, bins=bins, weights=all_mc_weight)
mc_variance, _ = np.histogram(all_mc_mass, bins=bins, weights=all_mc_weight**2)
mc_err = np.sqrt(mc_variance)

edges_rep = np.repeat(bins, 2)[1:-1]
lower_rep = np.repeat(mc_counts - mc_err, 2)
upper_rep = np.repeat(mc_counts + mc_err, 2)
ax.fill_between(edges_rep, lower_rep, upper_rep, step=None, color="gray",
                 alpha=0.4, hatch="///", edgecolor="none", label="MC stat. unc.")

data_names = [n for n, info in SAMPLES.items() if info["is_data"]]
data_mass = np.concatenate([results[n]["mass"] for n in data_names]) if data_names else np.array([])
data_counts, _ = np.histogram(data_mass, bins=bins)
data_err = np.sqrt(data_counts)

ax.errorbar(bin_centers, data_counts, yerr=data_err, fmt="ko", markersize=3, label="Data")

ax.set_ylabel("Events / 0.5 GeV")
ax.set_yscale("log")
ax.legend()
ax.set_title("Z → ee invariant mass")

with np.errstate(divide="ignore", invalid="ignore"):
    ratio = np.where(mc_counts > 0, data_counts / mc_counts, np.nan)
    ratio_err = np.where(mc_counts > 0, data_err / mc_counts, np.nan)
    mc_rel_err = np.where(mc_counts > 0, mc_err / mc_counts, np.nan)

lower_band = np.repeat(1 - mc_rel_err, 2)
upper_band = np.repeat(1 + mc_rel_err, 2)
ax_ratio.fill_between(edges_rep, lower_band, upper_band, color="gray", alpha=0.4, hatch="///")

ax_ratio.errorbar(bin_centers, ratio, yerr=ratio_err, fmt="ko", markersize=3)
ax_ratio.axhline(1.0, color="red", linestyle="--", linewidth=1)
ax_ratio.set_ylim(0.5, 1.5)
ax_ratio.set_ylabel("Data / MC")
ax_ratio.set_xlabel("$m_{ee}$ [GeV]")

plt.show()

In [ ]:
def combine_group(sample_names, syst_key=None):
    """Concatenate mass + (nominal or systematic) weight across a list of sample names."""
    masses_ = np.concatenate([results[n]["mass"] for n in sample_names])
    if syst_key is None:
        weights_ = np.concatenate([results[n]["weight"] for n in sample_names])
    else:
        weights_ = np.concatenate([results[n]["weight_syst"][syst_key] for n in sample_names])
    return masses_, weights_


def combine_group_dy(channel, syst_key=None):
    """Same as combine_group, but for a single DY channel slice (ee/mumu/tautau)."""
    dy_res = results["DY_inclusive_NLO"]
    mask = dy_res["channel"] == channel
    masses_ = dy_res["mass"][mask]
    if syst_key is None:
        weights_ = dy_res["weight"][mask]
    else:
        weights_ = dy_res["weight_syst"][syst_key][mask]
    return masses_, weights_


def combine_group_shape(sample_names, shape_key):
    masses_ = np.concatenate([results[n]["mass_syst"][shape_key] for n in sample_names])
    weights_ = np.concatenate([results[n]["weight"] for n in sample_names])
    return masses_, weights_


def combine_group_dy_shape(channel, shape_key):
    dy_res = results["DY_inclusive_NLO"]
    mask = dy_res["channel"] == channel
    return dy_res["mass_syst"][shape_key][mask], dy_res["weight"][mask]


def hist_from_counts(counts, bins):
    """Wrap a precomputed bin-count array (e.g. results[...]['pdf_up']) as a bh.Histogram,
    for writing out alongside the nominal ROOT histograms."""
    h = bh.Histogram(bh.axis.Variable(bins), storage=bh.storage.Weight())
    if counts is not None:
        h.view().value = counts
    return h


# group definitions -- match the grouping already used in the plotting cell
ttbar_names = [n for n, info in SAMPLES.items() if info.get("group") == "ttbar"]
wjets_names = [n for n, info in SAMPLES.items() if info.get("group") == "wjets"]
diboson_names = [n for n, info in SAMPLES.items() if info.get("group") == "diboson"]

# component_name -> (kind, sample_names_or_channel)
COMPONENTS = {
    "ttbar":   ("group", ttbar_names),
    "Wjets":   ("group", wjets_names),
    "Diboson": ("group", diboson_names),
    "DY_tautau": ("dy_channel", "tautau"),
    "DY_ee":     ("dy_channel", "ee"),
}

In [ ]:
def make_bh(mass, bins, weight=None):
    h = bh.Histogram(bh.axis.Variable(bins), storage=bh.storage.Weight())
    if weight is None:
        h.fill(mass)
    else:
        h.fill(mass, weight=weight)
    return h


OUT_DIR = "../datasets/z-ee"
os.makedirs(OUT_DIR, exist_ok=True)

for comp_name, (kind, spec) in COMPONENTS.items():
    fname = f"{OUT_DIR}/{comp_name.replace(chr(8594), '_').replace(' ', '_')}.root"

    with uproot.recreate(fname) as f:
        # nominal
        if kind == "group":
            mass_, weight_ = combine_group(spec, syst_key=None)
        else:
            mass_, weight_ = combine_group_dy(spec, syst_key=None)
        f["h_mass"] = make_bh(mass_, bins, weight_)

        # one-at-a-time weight systematics
        for syst in SYST_NAMES:
            if kind == "group":
                mass_s, weight_s = combine_group(spec, syst_key=syst)
            else:
                mass_s, weight_s = combine_group_dy(spec, syst_key=syst)
            f[f"h_mass_{syst}"] = make_bh(mass_s, bins, weight_s)

        for shift_key in SHAPE_SYST_NAMES:
            if kind == "group":
                mass_s, weight_s = combine_group_shape(spec, shift_key)
            else:
                mass_s, weight_s = combine_group_dy_shape(spec, shift_key)
            f[f"h_mass_{shift_key}"] = make_bh(mass_s, bins, weight_s)

        # PDF / scale envelopes -- only meaningful for the signal (DY_ee); extend to other
        # components too if you want their theory uncertainties in the fit as well.
        # NOTE: these are accumulated in process_sample over the *inclusive* DY sample, not
        # split by ee/mumu/tautau (the streaming accumulation happens before the channel split
        # in this version). Since mumu is ~0 and tautau feed-down is a small fraction after your
        # electron selection, this is dominated by ee -- fine as an approximation, but if you
        # want it exact, add a channel argument to _accumulate_envelope and keep one running
        # sum per channel inside the chunk loop.
        if comp_name == "DY_ee":
            dy_res = results["DY_inclusive_NLO"]
            f["h_mass_PDFUp"] = hist_from_counts(dy_res["pdf_up"], bins)
            f["h_mass_PDFDown"] = hist_from_counts(dy_res["pdf_down"], bins)
            f["h_mass_ScaleUp"] = hist_from_counts(dy_res["scale_up"], bins)
            f["h_mass_ScaleDown"] = hist_from_counts(dy_res["scale_down"], bins)

    print(f"wrote {fname} (nominal + {len(SYST_NAMES)} weight systematics"
          + (" + PDF/scale" if comp_name == "DY_ee" else "") + ")")

# --- Data (unweighted -> Poisson errors), unchanged ---
data_names = [n for n, info in SAMPLES.items() if info["is_data"]]
data_mass = np.concatenate([results[n]["mass"] for n in data_names]) if data_names else np.array([])
h_data = make_bh(data_mass, bins, weight=None)
with uproot.recreate(f"{OUT_DIR}/Data.root") as f:
    f["h_mass"] = h_data
